In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

project_root = Path.cwd()
if not (project_root / "articles").is_dir():
    project_root = project_root.parent
article_path = project_root / "articles"

documents = []
#rglob() searches recursively, so you don't need to manually specify every company folder.
for pdf_files in article_path.rglob("*.pdf"):
    loader = PyPDFLoader(str(pdf_files))
    documents.extend(loader.load())

Ignoring wrong pointing object 7 0 (offset 0)
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/AAAAAE+SFHello-Bold', '/FontDescriptor': IndirectObject(30, 0, 2018738393808), '/Encoding': '/MacRomanEncoding', '/FirstChar': 32, '/LastChar': 213, '/Widths': [224, 0, 0, 0, 0, 0, 0, 0, 417, 417, 0, 0, 0, 0, 303, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 303, 0, 0, 0, 0, 0, 0, 726, 675, 733, 729, 603, 578, 0, 765, 302, 0, 0, 574, 891, 746, 773, 655, 773, 677, 662, 636, 738, 0, 997, 0, 699, 0, 0, 0, 0, 0, 0, 0, 578, 0, 574, 633, 586, 0, 0, 0, 272, 0, 0, 278, 0, 612, 0, 629, 0, 0, 0, 396, 612, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 303]}, but is not installed. Consid

In [3]:
len(documents)

634

In [4]:
#here we break the data into small chunks of text
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

Total pages: 634
Total chunks: 2756


In [21]:
# Now convert the chunks into embeddings for LangChain Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

print("Number of chunks:", len(chunks))
print("Embedding dimension:", len(embedding_model.embed_query(chunks[0].page_content)))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2292.59it/s]


Number of chunks: 2756
Embedding dimension: 384


In [23]:
from langchain_chroma import Chroma
from types import SimpleNamespace

# Use the LangChain embedding object, not the precomputed `embedding` list

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="financial_documents"
)

In [24]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})
results = retriever.invoke("What was Apple's revenue in 2025?")

In [25]:
print(f"Length of reterivd docs : {len(results)}")

Length of reterivd docs : 5


In [26]:
results[4].page_content

'Apple Inc.\nCONSOLIDATED STATEMENTS OF OPERATIONS\n(In millions, except number of shares, which are reflected in thousands, and per-share amounts)\nYears ended\nSeptember 27,\n2025\nSeptember 28,\n2024\nSeptember 30,\n2023\nNet sales:\n   Products $ 307,003 $ 294,866 $ 298,085 \n   Services  109,158  96,169  85,200 \nTotal net sales  416,161  391,035  383,285 \nCost of sales:\n   Products  194,116  185,233  189,282 \n   Services  26,844  25,119  24,855 \nTotal cost of sales  220,960  210,352  214,137 \nGross margin  195,201  180,683  169,148 \nOperating expenses:\nResearch and development  34,550  31,370  29,915 \nSelling, general and administrative  27,601  26,097  24,932 \nTotal operating expenses  62,151  57,467  54,847 \nOperating income  133,050  123,216  114,301 \nOther income/(expense), net  (321)  269  (565) \nIncome before provision for income taxes  132,729  123,485  113,736 \nProvision for income taxes  20,719  29,749  16,741 \nNet income $ 112,010 $ 93,736 $ 96,995 \nEarni

In [48]:
#Creta LLM USING Gemini-3.6-flash
#from llama_index.llms.google_genai import GoogleGenAI
from dotenv import load_dotenv
import os

load_dotenv()

True

In [49]:
# Create a Chain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


system_prompt = """
You are an AI Financial Research Assistant specializing in stock and company analysis.

Answer the user's question using ONLY the information provided in the context.

Rules:
- Do not invent financial figures, facts, dates, or company information.
- If the context does not contain enough information, say:
  "The available documents do not contain enough information to answer this question."
- Preserve the original units such as USD, million, billion, and percentage.
- Clearly distinguish reported figures from calculated figures.
- When comparing financial periods, mention the relevant years or quarters.
- Explain financial terms briefly when useful.
- Do not provide personalized investment advice.
- Do not guarantee future stock performance or returns.

Provide your answer in this format:

Answer:
Give a concise answer to the user's question.

Key Financial Metrics:
Mention the relevant financial metrics and figures.

Analysis:
Explain the important trends or relationships found in the context.

Risks / Considerations:
Mention relevant risks or uncertainties found in the documents.

Source:
Mention the relevant document or page information when available.

-------------------------
Context:
{context}
-------------------------
"""


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


# Step 8: Create question-answer chain
# Use a LangChain LLM, not the LlamaIndex GoogleGenAI object
langchain_llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.4,
    max_tokens=1086,
)

question_answer_chain = create_stuff_documents_chain(
    langchain_llm,
    prompt
)


# Create RAG chain
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

In [44]:
response = rag_chain.invoke({
    "input": "What was the Tesla company's revenue in 2025?"
})

print(response["answer"])

c:\Users\ADITHYA UBALE\miniconda3\envs\fine-env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer:
In 2025, Tesla's total revenue was $94,827 million (or $94.83 billion), which represents a decrease of $2,863 million (or approximately 3% / $2.86 billion) compared to 2024.

Key Financial Metrics:
- **Total Revenues (2025):** $94,827 million ($94.83 billion)
  - **Total Automotive Revenues:** $69,526 million 
    - *Automotive sales:* $65,821 million
    - *Automotive regulatory credits:* $1,993 million
    - *Automotive leasing:* $1,712 million
  - **Energy Generation and Storage Segment Revenue:** $12,771 million
  - **Services and Other Revenue:** $12,530 million
- **Comparative Total Revenues:**
  - **2024:** $97,690 million
  - **2023:** $96,773 million

Analysis:
Total revenue declined by 3% in 2025 compared to 2024, primarily driven by a 10% decrease in total automotive revenues (falling from $77,070 million in 2024 to $69,526 million in 2025). Automotive sales revenue dropped by 9% ($6,659 million


In [46]:
context_docs = response.get("context", [])

if not context_docs:
    print("No context was returned by the retrieval chain.")
else:
    for doc in context_docs:
        if hasattr(doc, "page_content"):
            text = doc.page_content
            metadata = getattr(doc, "metadata", {})
        elif isinstance(doc, dict):
            text = doc.get("page_content") or doc.get("content") or str(doc)
            metadata = doc.get("metadata", {})
        else:
            text = str(doc)
            metadata = {}

        print(text[:500])
        print(metadata)
        print("-" * 50)


Tesla,	Inc.
Consolidated	Statements	of	Operations
(in	millions,	except	per	share	data)
Year	Ended	December	31,
2025
2024
2023
Revenues
Automotive	sales
$
65,821
	
$
72,480
	
$
78,509
	
Automotive	regulatory	credits
1,993
	
2,763
	
1,790
	
Automotive	leasing
1,712
	
1,827
	
2,120
	
Total	automotive	revenues
69,526
	
77,070
	
82,419
	
Energy	generation	and	storage
12,771
	
10,086
	
6,035
	
Services	and	other
12,530
	
10,534
	
8,319
	
Total	revenues
94,827
	
97,690
	
96,773
	
Cost	of	revenues
Autom
{'page': 52, 'title': '', 'page_label': '53', 'total_pages': 139, 'producer': 'Qt 5.15.8', 'creator': 'wkhtmltopdf 0.12.6', 'source': 'c:\\Users\\ADITHYA UBALE\\OneDrive\\Desktop\\FINANCIAL_STOCK_ANALYSIS_USING_LLAMA_INDEX\\articles\\Tesla\\Annual_Report_TSLA_2025.pdf', 'creationdate': '2026-01-29T11:10:47+00:00'}
--------------------------------------------------
In	2025,	we	recognized	total	revenues	of	$94.83	billion,	representing	a	decrease	of	$2.86	billion	compared	to	the	prior	year.	In	202